# W11C1 Lab: LoRA and Quantization, Measured

Run every cell from the top. **Everything already works.**

Today you will:

1. Count how many numbers a full fine-tune has to change.
2. Write LoRA yourself, in five lines, and count again.
3. Squeeze a model to 8 bits and measure exactly what you lose.

There is no test to run and nothing to submit. Each task tells you what
you should see when it is right.

In [ ]:
# Setup.
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)

# One layer from a realistically sized model: 1024 in, 1024 out.
IN, OUT = 1024, 1024
base_layer = nn.Linear(IN, OUT, bias=False)
for p in base_layer.parameters():
    p.requires_grad = False              # frozen, as a pretrained model would be

full_params = IN * OUT
print(f"one frozen layer: {IN} x {OUT} = {full_params:,} numbers")

## Part 1. The cost of fine-tuning everything

Full fine-tuning updates every number in the model, and you must store a
complete copy per task. LoRA's claim is that the UPDATE is simple even
when the model is not.

In [ ]:
# GIVEN. LoRA in five lines: learn two thin matrices instead of one fat one.
class LoRALayer(nn.Module):
    def __init__(self, base, rank=8, alpha=16):
        super().__init__()
        self.base = base                                   # frozen
        self.A = nn.Parameter(torch.randn(rank, base.in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(base.out_features, rank))
        self.scale = alpha / rank

    def forward(self, x):
        return self.base(x) + (x @ self.A.T @ self.B.T) * self.scale

lora = LoRALayer(base_layer, rank=8)
trainable = sum(p.numel() for p in lora.parameters() if p.requires_grad)

print(f"full fine-tune trains : {full_params:,} numbers")
print(f"LoRA rank 8 trains    : {trainable:,} numbers")
print(f"that is {100 * trainable / full_params:.2f}% of the model")
print()
print("B starts at ZERO, so B @ A is zero and the adapted model begins")
print("identical to the original. Training can only improve on it.")

In [ ]:
# GIVEN. Trainable parameters against rank, drawn.
ranks = [1, 2, 4, 8, 16, 32, 64]
counts = [r * (IN + OUT) for r in ranks]

plt.figure(figsize=(6.5, 3.2))
plt.plot(ranks, counts, "o-", color="#7C2529", label="LoRA")
plt.axhline(full_params, color="#999", linestyle="--", label="full fine-tune")
plt.xlabel("rank"); plt.ylabel("trainable numbers"); plt.yscale("log")
plt.legend(); plt.title("LoRA cost against rank"); plt.tight_layout(); plt.show()

for r, c in zip(ranks, counts):
    print(f"   rank {r:>2}: {c:>9,}  ({100 * c / full_params:5.2f}% of full)")

In [ ]:
# ================== YOUR TURN 1 ==================
# Can a low-rank update actually do the job? Fit the LoRA layer to a
# TARGET transformation and measure the error at different ranks.
#
# Try RANK = 1, then 4, then 32.
#
# Expected: on the low-rank update, rank 1 leaves 0.292 and rank 2 leaves 0.189,
#           but rank 4 hits 0.00000 EXACTLY and ranks 8 and 32 are no better. LoRA
#           needs exactly as much rank as the update actually has, and no more.
#           On the full-rank update it barely improves at any rank, which is the
#           method's limit: it works because real fine-tuning updates happen to be
#           low-rank, not because low-rank can fit anything.
# ===============================================
RANK = 1          # <-- try 2, then 4, then 32

torch.manual_seed(0)
x = torch.randn(256, IN)

# Two targets. The first needs a rank-4 update, which is what the LoRA paper
# claims real fine-tuning needs. The second needs a full-rank one.
U, V = torch.randn(IN, 4) * 0.1, torch.randn(4, OUT) * 0.1
low_rank_target = base_layer(x) + x @ U @ V
full_rank_target = base_layer(x) + x @ (torch.randn(IN, OUT) * 0.02)

def fit(target, rank):
    torch.manual_seed(0)
    layer = LoRALayer(base_layer, rank=rank)
    opt = torch.optim.Adam([q for q in layer.parameters() if q.requires_grad], lr=0.01)
    for step in range(400):
        opt.zero_grad()
        loss = ((layer(x) - target) ** 2).mean()
        loss.backward(); opt.step()
    return loss.item()

trainable = RANK * (IN + OUT)
print(f"rank {RANK}: {trainable:,} trainable numbers "
      f"({100 * trainable / full_params:.2f}% of full)")
print(f"   error on a genuinely LOW-RANK update : {fit(low_rank_target, RANK):.5f}")
print(f"   error on a FULL-RANK update          : {fit(full_rank_target, RANK):.5f}")

## Part 2. Quantization: fewer bits per number

The other way to make a model cheap is to store each number in less space.
32 bits down to 8 is a 4x saving. The question is what it costs you.

In [ ]:
# GIVEN. Quantize to 8 bits by hand, then measure the damage.
def quantize(tensor, bits=8):
    """Map the values onto a grid of 2**bits evenly spaced levels."""
    levels = 2 ** bits - 1
    lo, hi = tensor.min(), tensor.max()
    step = (hi - lo) / levels
    return torch.round((tensor - lo) / step) * step + lo

W = base_layer.weight.data
for bits in (8, 4, 2):
    Wq = quantize(W, bits)
    error = (W - Wq).abs().mean().item()
    size_mb = W.numel() * bits / 8 / 1e6
    print(f"   {bits:>2} bits: {size_mb:5.2f} MB, average error per weight {error:.5f}")
print(f"   32 bits: {W.numel() * 4 / 1e6:5.2f} MB, average error 0.00000  (the original)")

In [ ]:
# ================== YOUR TURN 2 ==================
# Measure what quantization does to the layer's OUTPUT, which is what
# actually matters.
#
# Try BITS = 8, then 4, then 2.
#
# Expected: 8 bits changes the output by well under a percent, which is why it is
#           the default everywhere. 4 bits is visible but often usable. 2 bits
#           wrecks it. Memory falls linearly with bits; quality falls off a cliff.
# ===============================================
BITS = 8          # <-- try 4, then 2

x = torch.randn(128, IN)
original = base_layer(x)

quantized_layer = nn.Linear(IN, OUT, bias=False)
quantized_layer.weight.data = quantize(base_layer.weight.data, BITS)
approx = quantized_layer(x)

relative = ((original - approx).norm() / original.norm()).item()
print(f"{BITS} bits")
print(f"   memory      : {100 * BITS / 32:.0f}% of the original")
print(f"   output error: {100 * relative:.2f}%")

## Answers

Try each task before reading.

In [ ]:
# YOUR TURN 1
#   Against the rank-4 target: 0.292, 0.189, 0.00000, 0.00000, 0.00000 for
#   ranks 1, 2, 4, 8, 32. The error hits zero the moment the adapter has as
#   much rank as the update, and extra rank is wasted.
#
#   Against the full-rank target it never gets there. That is the honest
#   limit of the method: LoRA works because the updates real fine-tuning
#   needs turn out to be low-rank, which is an empirical claim about tasks,
#   not a mathematical guarantee.
#
# YOUR TURN 2
#   8 bits costs a fraction of a percent of output accuracy for a 4x memory
#   saving, which is why it is the default. 2 bits destroys the layer. Note that
#   LoRA and quantization compose: quantize the frozen base, train a small
#   adapter on top, and you have QLoRA.